# Synthetic Minority Oversampling Technique (SMOTE)

# Algorithm

<img src="image/smote-1.png" width="1600">

<img src="image/smote-3.png" width="1600">

<img src="image/smote_fig.png" width="1600">

<img src="image/smote-2.png" width="1600">

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, accuracy_score

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

import warnings
warnings.filterwarnings("ignore")

In [ ]:
X, y = make_classification(
    n_samples=2000,
    n_features=2,          # keep 2D so we can plot
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    weights=[0.95, 0.05],  # 95% vs 5% (highly imbalanced)
    class_sep=1.0,
    random_state=42
)

print("Original class counts:", np.bincount(y))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

print("Train class counts:", np.bincount(y_train))
print("Test  class counts:", np.bincount(y_test))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# -------- Before SMOTE --------
axes[0].scatter(
    X_train[y_train == 0, 0],
    X_train[y_train == 0, 1],
    s=10, alpha=0.6, label="Class 0 (majority)"
)
axes[0].scatter(
    X_train[y_train == 1, 0],
    X_train[y_train == 1, 1],
    s=20, alpha=0.8, label="Class 1 (minority)"
)
axes[0].set_title("Training data (Before SMOTE)")
axes[0].legend()

# -------- Apply SMOTE --------
from imblearn.over_sampling import SMOTE
smote = SMOTE(k_neighbors=5, random_state=42)
X_res, y_res = smote.fit_resample(X_train, y_train)

# -------- After SMOTE --------
axes[1].scatter(
    X_res[y_res == 0, 0],
    X_res[y_res == 0, 1],
    s=10, alpha=0.4, label="Class 0 (majority)"
)
axes[1].scatter(
    X_res[y_res == 1, 0],
    X_res[y_res == 1, 1],
    s=10, alpha=0.4, label="Class 1 (minority + synthetic)"
)
axes[1].set_title("Training data (After SMOTE)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
baseline = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

print("=== Baseline (no SMOTE) ===")
print("Class 0 ->",
      "Precision:", f"{precision_score(y_test, y_pred_base, pos_label=0):.4f}",
      "Recall:", f"{recall_score(y_test, y_pred_base, pos_label=0):.4f}")

print("Class 1 ->",
      "Precision:", f"{precision_score(y_test, y_pred_base):.4f}",
      "Recall:", f"{recall_score(y_test, y_pred_base):.4f}")

print("Accuracy:", f"{accuracy_score(y_test, y_pred_base):.4f}")


smote_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("smote", SMOTE(k_neighbors=5, random_state=42)),  # applied only during fit on train folds
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

smote_model.fit(X_train, y_train)
y_pred_smote = smote_model.predict(X_test)

print("\n=== SMOTE ===")
print("Class 0 ->",
      "Precision:", f"{precision_score(y_test, y_pred_smote, pos_label=0):.4f}",
      "Recall:", f"{recall_score(y_test, y_pred_smote, pos_label=0):.4f}")

print("Class 1 ->",
      "Precision:", f"{precision_score(y_test, y_pred_smote):.4f}",
      "Recall:", f"{recall_score(y_test, y_pred_smote):.4f}")

print("Accuracy:", f"{accuracy_score(y_test, y_pred_smote):.4f}")

In [ ]:
def plot_decision_boundary(model, X, y, ax, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid = np.c_[xx.ravel(), yy.ravel()]
    Z = model.predict(grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.25)
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=10, alpha=0.6, label="Class 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=20, alpha=0.8, label="Class 1")
    ax.set_title(title)
    ax.legend()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

plot_decision_boundary(
    baseline, X_test, y_test, axes[0], "Decision Boundary (Baseline)"
)

plot_decision_boundary(
    smote_model, X_test, y_test, axes[1],"Decision Boundary (With SMOTE)"
)

plt.tight_layout()
plt.show()